In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
load_dotenv()




True

In [5]:
##Document loader
yt_api = YouTubeTranscriptApi()
video_id = 'ZhAz268Hdpw'
transcript = yt_api.fetch(video_id)


In [10]:
transcript
transcript_text = " ".join(doc.text for doc in transcript)
transcript_text


"Chad GPT is powered by a model called GPT which is based on a deep learning architecture called Transformers Transformers is the reason behind modern day AI boom as an AI Enthusiast when you start learning Transformers you will come across this complex diagram which will start giving you a headache immediately my goal for today's video is to explain you Transformers in a most simplified and intuitive manner we need to cover many different topics so this is going to be a long video attention and patience is all I need from you today when you type in a sentence in Gmail it tries to predict next word or next set of words this is possible because of a machine learning model called language model Google for example has this popular language model called bird which is powering hundreds of AI applications throughout the world GPT which is a model behind chat GPT is a large language model the reason it is called large language model is because it has billions of parameters it is much more cap

In [ ]:
# from h11._abnf import chunk_size
#text splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks =splitter.split_text(transcript_text)
chunks


105

In [13]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')


In [16]:
#vector store
store = FAISS.from_texts(chunks,embeddings)

In [17]:
store.similarity_search(query="tell me something about transformers")

[Document(id='9e319684-3030-466e-b242-5f920b162982', metadata={}, page_content="so that's that was about Transformer I know it was a long discussion there were many topics that we covered but hopefully your understanding is clear if you have any question please feel free to ask [Music] a [Music]"),
 Document(id='655efe30-9ee0-47fc-ab2b-cb8c38f8a29b', metadata={}, page_content="Chad GPT is powered by a model called GPT which is based on a deep learning architecture called Transformers Transformers is the reason behind modern day AI boom as an AI Enthusiast when you start learning Transformers you will come across this complex diagram which will start giving you a headache immediately my goal for today's video is to explain you Transformers in a most simplified and intuitive manner we need to cover many different topics so this is going to be a long video attention and"),
 Document(id='520e5da7-4fbf-4948-b51a-d5bd8d2a936e', metadata={}, page_content="Channel called 3 blue one brown so if

In [18]:
retreiever = store.as_retriever(search_kwargs={"k":4})

In [19]:
retreiever.invoke("Explain about chatgpt")

[Document(id='655efe30-9ee0-47fc-ab2b-cb8c38f8a29b', metadata={}, page_content="Chad GPT is powered by a model called GPT which is based on a deep learning architecture called Transformers Transformers is the reason behind modern day AI boom as an AI Enthusiast when you start learning Transformers you will come across this complex diagram which will start giving you a headache immediately my goal for today's video is to explain you Transformers in a most simplified and intuitive manner we need to cover many different topics so this is going to be a long video attention and"),
 Document(id='a651229a-689c-4016-868d-b2267c55bfcb', metadata={}, page_content='it is called large language model is because it has billions of parameters it is much more capable and Powerful compared to bird and it is trained on humongous amount of data fundamentally though it is also doing the same thing which is when you type in a question in chat GPT it will predict the next word in that sentence and then it w

In [20]:
prompt = PromptTemplate(template="""You are a Helpful assistant. Answer from the following context.
if the context is insufficient then directly sat I don't know.
{context}
Question {question}""",
input_variables=['context','question'])

In [27]:
question = "Explain about finance?"

In [28]:
context = retreiever.invoke(question)

context = " ".join(i.page_content for i in context)

In [29]:
final_prompt = prompt.invoke({"question":question,"context":context})

In [30]:
#generation
llm = ChatOpenAI(model='gpt-3.5-turbo',temperature=.2)

In [31]:
answer = llm.invoke(final_prompt)
answer.content

"I don't know."